In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!unzip '/content/drive/MyDrive/Audio Signal Dataset for Danger Detection of Women and Children.zip'

Archive:  /content/drive/MyDrive/Audio Signal Dataset for Danger Detection of Women and Children.zip
replace Audio Signal Dataset for Danger Detection of Women┬аand┬аChildren/test/Child/1103_4_patch.wav? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt

# Load a sample audio file
audio_path = '/content/Audio Signal Dataset for Danger Detection of Women┬аand┬аChildren/train/Child/1002_0.wav'
y, sr = librosa.load(audio_path)

# Display the waveform
plt.figure(figsize=(14, 5))
librosa.display.waveshow(y, sr=sr)
plt.title('Waveform')
plt.show()

print(f'Audio loaded successfully. Sample rate: {sr} Hz')

In [ ]:
import os
import pandas as pd

# Define the root directory where your folders are stored
root_dir = '/content/Audio Signal Dataset for Danger Detection of Women┬аand┬аChildren/train'  # <-- replace this with your actual path

# Define your classes
categories = ['Women', 'Child', 'Normal']

# Prepare a list to collect data
data = []

# Loop through each category folder
for category in categories:
    folder_path = os.path.join(root_dir, category)
    for filename in os.listdir(folder_path):
        if filename.endswith('.wav'):
            file_path = os.path.join(folder_path, filename)
            data.append({'file_path': file_path, 'label': category})

# Create a DataFrame
df = pd.DataFrame(data)

# Save to Excel
output_path = os.path.join(root_dir, 'audio_dataset_labels.xlsx')
df.to_excel(output_path, index=False)

print(f"Excel sheet saved at: {output_path}")

In [ ]:
import os
import librosa
import numpy as np
import pandas as pd

# Define your dataset directory
root_dir = '/content/Audio Signal Dataset for Danger Detection of Women┬аand┬аChildren/train'  # <- replace with your actual path
categories = ['Women', 'Child', 'Normal']

# List to store the final data
data = []

# Feature extraction function
def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=None)

    features = {
        "zcr": np.mean(librosa.feature.zero_crossing_rate(y)),
        "rms": np.mean(librosa.feature.rms(y=y)),
        "spectral_centroid": np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)),
    }

    # MFCCs: We take mean of first 13 coefficients
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i in range(13):
        features[f'mfcc_{i+1}'] = np.mean(mfccs[i])

    return features

# Loop through each category and each audio file
for label in categories:
    folder_path = os.path.join(root_dir, label)
    for file in os.listdir(folder_path):
        if file.endswith('.wav'):
            file_path = os.path.join(folder_path, file)
            try:
                features = extract_features(file_path)
                features['label'] = label
                data.append(features)
            except Exception as e:
                print(f"Error processing {file_path}: {e}")

# Create DataFrame
df = pd.DataFrame(data)

# Save to Excel (optional)
df.to_excel(os.path.join(root_dir, 'audio_features_dataset.xlsx'), index=False)

print("Feature extraction complete. Excel file saved.")

In [ ]:
from sklearn.model_selection import train_test_split

# Separate features (X) and labels (y)
X = df[inputs]
y = df['label']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data split into training and testing sets.")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model on the training data
model.fit(X_train, y_train)

# Predict on the test data
y_pred = model.predict(X_test)

print("Random Forest model trained successfully.")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

# Initialize and train the Logistic Regression model
log_reg_model = LogisticRegression(max_iter=1000, random_state=42) # Increased max_iter for convergence
log_reg_model.fit(X_train, y_train)

# Predict and evaluate Logistic Regression
y_pred_lr = log_reg_model.predict(X_test)
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f"Accuracy of the Logistic Regression model: {accuracy_lr:.4f}")
print("\nClassification Report (Logistic Regression):\n", classification_report(y_test, y_pred_lr))

print("-" * 30) # Separator for clarity

# Initialize and train the Gaussian Naive Bayes model
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

# Predict and evaluate Naive Bayes
y_pred_nb = nb_model.predict(X_test)
accuracy_nb = accuracy_score(y_test, y_pred_nb)
print(f"Accuracy of the Gaussian Naive Bayes model: {accuracy_nb:.4f}")
print("\nClassification Report (Gaussian Naive Bayes):\n", classification_report(y_test, y_pred_nb))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Generate confusion matrix for Random Forest
cm_rf = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', xticklabels=model.classes_, yticklabels=model.classes_)
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# Generate confusion matrix for Logistic Regression
cm_lr = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', xticklabels=log_reg_model.classes_, yticklabels=log_reg_model.classes_)
plt.title('Confusion Matrix - Logistic Regression')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# Generate confusion matrix for Gaussian Naive Bayes
cm_nb = confusion_matrix(y_test, y_pred_nb)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues', xticklabels=nb_model.classes_, yticklabels=nb_model.classes_)
plt.title('Confusion Matrix - Gaussian Naive Bayes')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
!pip install transformers torch matplotlib seaborn